# Exercise 51 — Parquet

## Concept

Parquet is a **columnar, compressed, typed** file format. The standard for storing tabular data on disk in modern data platforms.

Why not CSV?
- **Typed**: integers, dates, booleans preserved — no "`"42"` vs `42`" parsing.
- **Compressed**: typically 5–20× smaller than CSV.
- **Columnar**: reading just 3 columns from a 200-column file scans only those columns.
- **Schema**: every file embeds its schema. No guessing.

Why not always Parquet?
- Binary — not human-readable, not appendable by simple file ops.
- Needs a library (`pyarrow`).

### With pandas (most common entry point)

```python
import pandas as pd

df.to_parquet("data.parquet", index=False)
df = pd.read_parquet("data.parquet")

# Read only a subset of columns — leverage the columnar layout
df = pd.read_parquet("data.parquet", columns=["id", "amount"])
```

### Setup requirement

This exercise needs `pyarrow` (the engine pandas uses for Parquet by default in recent versions):

```
pip install pandas pyarrow
```

If the import fails, run `!pip install pandas pyarrow` in a notebook cell and retry.

## Setup

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.DataFrame({
    "order_id":   [1001, 1002, 1003, 1004, 1005],
    "customer":   ["Alice", "Bob", "Alice", "Carol", "Bob"],
    "amount":     [120.50, 45.00, 300.75, 89.99, 210.00],
    "paid":       [True, True, False, True, False],
})
print(df)


## Task 1 — Write and read back

Write a function that saves the DataFrame to a Parquet file, then a function that reads it back. Verify a round-trip produces an equal DataFrame.

```python
def save_parquet(df: pd.DataFrame, path: str) -> None: ...
def load_parquet(path: str) -> pd.DataFrame: ...
```

Use `df.to_parquet(path, index=False)` — `index=False` is the right default for tabular data (no row labels added).

In [ ]:
import pandas as pd

def save_parquet(df: pd.DataFrame, path: str) -> None:
    pass  # TODO

def load_parquet(path: str) -> pd.DataFrame:
    pass  # TODO

save_parquet(df, "orders.parquet")
loaded = load_parquet("orders.parquet")
assert loaded.equals(df)
assert loaded["amount"].dtype == df["amount"].dtype
assert loaded["paid"].dtype == bool                # type preserved
print("ok")


## Task 2 — Read selected columns

Write a function that reads only the `order_id` and `amount` columns from a Parquet file.

```python
def load_id_amount(path: str) -> pd.DataFrame:
    ...
```

Use the `columns=` argument of `read_parquet`. With a real-world 200-column file this is dramatically faster than loading everything.

In [ ]:
import pandas as pd

def load_id_amount(path: str) -> pd.DataFrame:
    pass  # TODO

sub = load_id_amount("orders.parquet")
assert list(sub.columns) == ["order_id", "amount"]
assert len(sub) == 5
print("ok")


## Task 3 — Compare file sizes vs CSV

Save the same DataFrame as CSV and as Parquet. Return both file sizes in bytes, and assert that Parquet is **smaller** than CSV.

```python
def compare_sizes(df: pd.DataFrame) -> tuple[int, int]:
    """Return (csv_size_bytes, parquet_size_bytes)."""
```

The difference is modest on tiny data — but on millions of rows with low-cardinality columns it's typically 10× or more. (`Path(path).stat().st_size` gives the byte size.)

In [ ]:
from pathlib import Path
import pandas as pd

def compare_sizes(df: pd.DataFrame) -> tuple[int, int]:
    pass  # TODO

csv_size, pq_size = compare_sizes(df)
print(f"csv={csv_size}B  parquet={pq_size}B")
assert csv_size > 0 and pq_size > 0
assert pq_size < csv_size      # Parquet wins even on this tiny example
print("ok")


## Task 4 — Schema preservation

Demonstrate that Parquet preserves types where CSV loses them.

1. Save `df` to both `roundtrip.csv` (no dtype hints) and `roundtrip.parquet`.
2. Read both back.
3. Compare the dtype of the `paid` column.

Return a tuple `(csv_paid_dtype_name, parquet_paid_dtype_name)` and assert:
- The Parquet version is `"bool"`.
- The CSV version is **not** `"bool"` (typically `"object"` — strings `"True"`/`"False"`).

```python
def compare_paid_dtypes(df: pd.DataFrame) -> tuple[str, str]:
    ...
```

In [ ]:
import pandas as pd

def compare_paid_dtypes(df: pd.DataFrame) -> tuple[str, str]:
    pass  # TODO

csv_dt, pq_dt = compare_paid_dtypes(df)
print("CSV dtype:", csv_dt, "  Parquet dtype:", pq_dt)
assert pq_dt == "bool"
assert csv_dt != "bool"
print("ok")
